In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All"
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

#import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


## IMPORTS

In [2]:
import random
import re
from collections import Counter

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupKFold

import torch
import torch.nn as nn
import torch.nn.functional as functional
from torch.optim import Adam, AdamW
from torch.utils.data import DataLoader
from transformers import AutoModel, AutoModelForMultipleChoice, AutoTokenizer, get_linear_schedule_with_warmup

In [3]:
FINAL_MODEL = "deberta"
#FINAL_MODEL = "minilm"
# FINAL_MODEL = "bilstm"
# FINAL_MODEL = "bert_lr"
# FINAL_MODEL = "tfidf_lr"

TRAIN_PATH = "/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv"
TEST_PATH = "/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv"

SEED = 33
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = 'cpu'
if torch.cuda.is_available():
    device = 'cuda'
    torch.cuda.manual_seed_all(SEED)

# EXPLORATORY DATA ANALYSIS

In [4]:
df = pd.read_csv(TRAIN_PATH)

In [5]:
df.head()

,id,prompt,A,B,C,D,E,answer
0,1,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
1,2,What is accelerator-based light-ion fusion?,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,A
2,3,Determine the correct option: What is the term...,Blueshifting,Redshifting,Reddening,Whitening,Yellowing,C
3,4,Select the most accurate option: What is Marti...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
4,5,Identify the correct statement: What is the co...,"Simultaneity is relative, meaning that two eve...","Simultaneity is relative, meaning that two eve...","Simultaneity is absolute, meaning that two eve...",Simultaneity is a concept that applies only to...,Simultaneity is a concept that applies only to...,A


In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 8 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   id      2000 non-null   int64 
 1   prompt  2000 non-null   object
 2   A       2000 non-null   object
 3   B       2000 non-null   object
 4   C       2000 non-null   object
 5   D       2000 non-null   object
 6   E       2000 non-null   object
 7   answer  2000 non-null   object
dtypes: int64(1), object(7)
memory usage: 125.1+ KB


In [7]:
df['prompt'].str.len()

0       142
1        43
2       182
3       129
4       110
       ... 
1995     72
1996    104
1997     93
1998    266
1999    140
Name: prompt, Length: 2000, dtype: int64

In [8]:
df['prompt'].str.len().sum()/2000

np.float64(117.6695)

In [9]:
df['prompt'].str.count(':').value_counts().sort_index()

prompt
0     480
1    1511
2       9
Name: count, dtype: int64

In [10]:
df.loc[df['prompt'].str.count(':') == 2, 'prompt']

331     Identify the correct statement: What is the pu...
450     Identify the correct statement: What is the pu...
845     Determine the correct option: What is the purp...
1593    Determine the correct option: What is the purp...
1684    Pick the best possible answer: What is the pur...
1796    Select the most accurate option: What is the p...
1835    Select the most accurate option: What is the p...
1854    Pick the best possible answer: What is the pur...
1913    Pick the best possible answer: What is the pur...
Name: prompt, dtype: object

In [11]:
for i, prompt in enumerate(df.loc[df['prompt'].str.count(':') == 2, 'prompt'], 1):
    print(i, prompt, '\n')

1 Identify the correct statement: What is the purpose of expressing a map's scale as a ratio, such as 1:10,000? carefully. 

2 Identify the correct statement: What is the purpose of expressing a map's scale as a ratio, such as 1:10,000? among the listed options. 

3 Determine the correct option: What is the purpose of expressing a map's scale as a ratio, such as 1:10,000? from the following choices. 

4 Determine the correct option: What is the purpose of expressing a map's scale as a ratio, such as 1:10,000? based on the given context. 

5 Pick the best possible answer: What is the purpose of expressing a map's scale as a ratio, such as 1:10,000? from the following choices. 

6 Select the most accurate option: What is the purpose of expressing a map's scale as a ratio, such as 1:10,000? from the following choices. 

7 Select the most accurate option: What is the purpose of expressing a map's scale as a ratio, such as 1:10,000? 

8 Pick the best possible answer: What is the purpose of 

In [12]:
df['prompt'].str.count(r'\?').value_counts().sort_index()

prompt
1    1718
2     282
Name: count, dtype: int64

In [13]:
df.loc[df['prompt'].str.count(r'\?') == 2, 'prompt']

6       Which of the following is correct? What is a "...
9       Which of the following is correct? What is the...
19      Which of the following is correct? Who propose...
21      Which of the following is correct? Who shared ...
23      Which of the following is correct? Which of th...
                              ...                        
1948    Which of the following is correct? What is res...
1971    Which of the following is correct? What is the...
1975    Which of the following is correct? What is the...
1978    Which of the following is correct? What is the...
1991    Which of the following is correct? What is a t...
Name: prompt, Length: 282, dtype: object

In [14]:
for i, prompt in enumerate(df.loc[df['prompt'].str.count(r'\?') == 2, 'prompt'].head(10), 1):
    print(i, prompt, '\n')

1 Which of the following is correct? What is a "coffee ring" in physics? among the listed options. 

2 Which of the following is correct? What is the relationship between the Wigner function and the density matrix operator? from the following choices. 

3 Which of the following is correct? Who proposed the concept of "maximal acceleration"? carefully. 

4 Which of the following is correct? Who shared the other half of the Nobel Prize with Yoichiro Nambu for discovering the origin of the explicit breaking of CP symmetry in the weak interactions? among the listed options. 

5 Which of the following is correct? Which of the following statements accurately describes the blocking temperature of an antiferromagnetic layer in a spin valve? carefully. 

6 Which of the following is correct? What is the Ramsauer-Townsend effect? carefully. 

7 Which of the following is correct? What is the reason for the stochastic nature of all observed resistance-switching processes? from the following choices

In [15]:
df.loc[df['prompt'].str.count(r'\?') == 1, 'prompt']

0       Pick the best possible answer: What is Martin ...
1             What is accelerator-based light-ion fusion?
2       Determine the correct option: What is the term...
3       Select the most accurate option: What is Marti...
4       Identify the correct statement: What is the co...
                              ...                        
1995    What is the piezoelectric strain coefficient f...
1996    Identify the correct statement: What is the sy...
1997    Determine the correct option: What does Earnsh...
1998    Identify the correct statement: What is the re...
1999    Pick the best possible answer: Which of the fo...
Name: prompt, Length: 1718, dtype: object

In [16]:
prefixes, questions, suffixes = [], [], []

def clean(prefixes, questions, suffixes, df):
    for text in df['prompt']:
        text = text.strip()
        prefix, suffix = '', ''

        ci = text.find(': ')
        if ci != -1:
            prefix = text[:ci].strip()
            text = text[ci + 1:].strip()

        n_q = text.count('?')

        if n_q == 1:
            qi = text.index('?')
            suffix = text[qi + 1:].strip()
            text = text[:qi + 1].strip()

        elif n_q == 2:
            first_q = text.index('?')
            prefix = text[:first_q + 1].strip()

            text = text[first_q + 1:].strip()
            second_q = text.index('?')

            suffix = text[second_q + 1:].strip()
            text = text[:second_q + 1].strip()

        prefixes.append(prefix or None)
        questions.append(text)
        suffixes.append(suffix or None)

clean(prefixes, questions, suffixes, df)

df['prefix'] = prefixes
df['question'] = questions
df['suffix'] = suffixes

print(df['prefix'].value_counts(dropna=False))
print(df['suffix'].value_counts(dropna=False))

prefix
Pick the best possible answer         316
Select the most accurate option       309
Identify the correct statement        302
Determine the correct option          299
Choose the correct answer             292
Which of the following is correct?    282
None                                  200
Name: count, dtype: int64
suffix
None                           538
among the listed options.      388
based on the given context.    376
from the following choices.    369
carefully.                     329
Name: count, dtype: int64


In [17]:
df['prompt'].duplicated().sum()

np.int64(242)

In [18]:
df['question'].duplicated().sum()

np.int64(1748)

In [19]:
df.groupby('question')['answer'].nunique().max()

1

In [20]:
df['correct_text'] = df.apply(lambda r: r[r['answer']], axis=1)

df.groupby('question')['correct_text'].nunique().max()

9

In [21]:
df.groupby('question')['correct_text'].nunique().sort_values(ascending=False).head(20)

question
What is the Erlangen program?                                                                                                                                                      9
What is the Maxwell's Demon thought experiment?                                                                                                                                    8
What is the butterfly effect, as defined by Lorenz in his book "The Essence of Chaos"?                                                                                             6
What is accelerator-based light-ion fusion?                                                                                                                                        4
What is radiometric dating?                                                                                                                                                        4
What is the application of Memristor?                                                 

In [22]:
df.groupby('question').size().value_counts().sort_index()

1     30
2     15
3      6
4     13
5     12
6      9
7     20
8     25
9     23
10    21
11    23
12    21
13    18
14     3
15     3
16     6
18     2
20     1
21     1
Name: count, dtype: int64

In [23]:
df.groupby('question')['question'].nunique().head(20)

question
How do the Lunar Laser Ranging Experiment, radar astronomy, and the Deep Space Network determine distances to the Moon, planets, and spacecraft?    1
How many crystallographic point groups are there in three-dimensional space?                                                                        1
What are amorphous ferromagnetic metallic alloys, and what are their advantages?                                                                    1
What are coherent turbulent structures?                                                                                                             1
What are permutation-inversion groups?                                                                                                              1
What are some models that attempt to account for all observations without invoking supplemental non-baryonic matter?                                1
What are the Navier-Stokes equations?                                                      

In [24]:
def option_set(row):
    return frozenset(row[o] for o in ['A', 'B', 'C', 'D', 'E'])

df['option_set'] = df.apply(option_set, axis=1)

unique_sets_per_question = df.groupby('question')['option_set'].nunique()
unique_sets_per_question.value_counts().sort_index()

option_set
1     187
2       8
3       1
4      14
5       6
6       7
7       5
8       7
9       2
10      3
11      5
12      4
13      1
15      1
16      1
Name: count, dtype: int64

In [25]:
varying_questions = unique_sets_per_question[unique_sets_per_question > 1]

print(len(varying_questions), 'of', len(unique_sets_per_question), 'unique questions have more than one distinct option set')
varying_questions.sort_values(ascending=False).head(10)

65 of 252 unique questions have more than one distinct option set


question
What is the Maxwell's Demon thought experiment?                                                                                                                                                                   16
Which of the following statements accurately describes the impact of Modified Newtonian Dynamics (MOND) on the observed "missing baryonic mass" discrepancy in galaxy clusters?                                   15
What is the Evans balance?                                                                                                                                                                                        13
What is the second law of thermodynamics?                                                                                                                                                                         12
What is the butterfly effect, as defined by Lorenz in his book "The Essence of Chaos"?                                                     

In [26]:
option_set_sizes = df.groupby('option_set').size()
repeated_sets = option_set_sizes[option_set_sizes > 1]
correct_per_set = df.groupby('option_set')['correct_text'].nunique()

print('option sets that repeat:', len(repeated_sets))
print('single correct answer:', (correct_per_set[repeated_sets.index] == 1).sum())
print('multiple correct answer:', (correct_per_set[repeated_sets.index] > 1).sum())

option sets that repeat: 210
single correct answer: 210
multiple correct answer: 0


In [27]:
df['answer'].value_counts(normalize=True).sort_index()

answer
A    0.1845
B    0.2450
C    0.2295
D    0.1790
E    0.1620
Name: proportion, dtype: float64

In [28]:
df.groupby('prefix')['answer'].value_counts(normalize=True).unstack().round(3)

answer,A,B,C,D,E
prefix,,,,,
Choose the correct answer,0.182,0.247,0.243,0.171,0.158
Determine the correct option,0.177,0.221,0.224,0.187,0.191
Identify the correct statement,0.159,0.238,0.238,0.192,0.172
Pick the best possible answer,0.206,0.247,0.206,0.193,0.149
Select the most accurate option,0.201,0.246,0.227,0.172,0.155
Which of the following is correct?,0.181,0.277,0.248,0.149,0.145


In [29]:
def length_rank(row):
    lengths = {opt: len(str(row[opt]).split()) for opt in ['A', 'B', 'C', 'D', 'E']}
    order = sorted(lengths, key=lambda opt: lengths[opt])
    return order.index(row['answer']) + 1

df['correct_len_rank'] = df.apply(length_rank, axis=1)
df['correct_len_rank'].value_counts(normalize=True).sort_index()

correct_len_rank
1    0.1560
2    0.1020
3    0.2140
4    0.1745
5    0.3535
Name: proportion, dtype: float64

In [30]:
df['correct_len'] = df.apply(lambda r: len(str(r[r['answer']]).split()), axis=1)
df['wrong_len'] = df.apply(lambda r: np.mean([len(str(r[opt]).split()) for opt in ['A', 'B', 'C', 'D', 'E'] if opt != r['answer']]), axis=1)

print('avg correct answer words:', df['correct_len'].mean())
print('avg wrong option words:', df['wrong_len'].mean())

avg correct answer words: 28.659
avg wrong option words: 25.68475


In [31]:
test_df = pd.read_csv(TEST_PATH)

test_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 7 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   id      500 non-null    int64 
 1   prompt  500 non-null    object
 2   A       500 non-null    object
 3   B       500 non-null    object
 4   C       500 non-null    object
 5   D       500 non-null    object
 6   E       500 non-null    object
dtypes: int64(1), object(6)
memory usage: 27.5+ KB


In [32]:
test_prefixes, test_questions, test_suffixes = [], [], []

clean(test_prefixes, test_questions, test_suffixes, test_df)

test_df['prefix'] = test_prefixes
test_df['question'] = test_questions
test_df['suffix'] = test_suffixes

print('train unique questions:', df['question'].nunique())
print('test unique questions:', test_df['question'].nunique())
print('test questions also seen in train:', test_df['question'].isin(df['question']).sum())

train unique questions: 252
test unique questions: 228
test questions also seen in train: 500


In [33]:
test_df['option_set'] = test_df.apply(option_set, axis=1)
train_sets_by_question = df.groupby('question')['option_set'].apply(set)

exact_matches = 0

for _, row in test_df.iterrows():
    train_sets = train_sets_by_question.get(row['question'])
    if train_sets is not None and row['option_set'] in train_sets:
        exact_matches += 1

print('test rows whose all options exactly match a train row same question:', exact_matches, '/', len(test_df))

test rows whose all options exactly match a train row same question: 455 / 500


## EDA LEARNINGS.MD

### Data
- Columns in train dataset: `id`, `prompt`, options `A` to `E`, and `answer`.
- Average prompt length is about 118 characters.

### Prompt
- Turns out the prompt has prefix, actual question and the suffix.

There are only 6 prefixes and 4 suffixes:
- Prefixes: Pick the best possible answer / Select the most accurate option / Identify the correct statement / Determine the correct option / Choose the correct answer / Which of the following is correct? (~200 prompts have no prefix)
- Suffixes: among the listed options / based on the given context / from the following choices / carefully (~538 prompts have no suffix)

### Duplicates
- Only 242 full prompts are exact duplicates.
- But once I strip the prefix and suffix, 1748 of the questions are duplicates. So most questions repeat, they were just dressed up with different prefixes/suffixes.
- There remains 252 unique questions out of 2000.
- Some questions are repeated once(30) while others are repeated 20 times.
- From those 252 unique questions, 187 of them have exact same options(found using frozen set).

### Same question, but the answer text changes
- Each question always maps to the same answer letter across duplicates.
- But the text of the correct answer varies. The same question can have up to 9 different versions of the correct answer text.
- Example: "What is the Erlangen program?" has 9 different correct answer texts.
- The wording changes by small margins, similar words used.

### Distribution of answer label
- B, C are over represented(~combined 47.5%), while E is under ~16.2%
- Usually longest option is more likely to be correct ~35%

### Overwhelming repeatition of questions in test dataset
- All 500 questions of test dataset are present in train dataset.
- There are 228 unique question, so 272 duplicates after processing.
- And 455 out of those 500 questions are exact overlap with same options as of train dataset.

*Theoretically this means I can score 0.95-0.97(0.91 directly correct and 0.75 from BiLSTM) MAP@3 score on kaggle competition by using current BiLSTM model as fallback model and primarily using lookback only, though I wonder how much it would hold up on private dataset. Worth a try ;).*

### Data leakage

Issue of question leakage where a single word has been changed.  
`What happens to an initially inhomogeneous physical framework that is isolated by a thermodynamic operation?
What happens to an initially inhomogeneous physical mechanism that is isolated by a thermodynamic operation?
What happens to an initially inhomogeneous physical structure that is isolated by a thermodynamic operation?`

# UTILS


In [34]:
OPTIONS = ["A", "B", "C", "D", "E"]

def split_prompt(prompt):
    text = str(prompt).strip()
    prefix, suffix = '', ''

    ci = text.find(': ')
    if ci != -1:
        prefix = text[:ci].strip()
        text = text[ci + 1:].strip()

    n_q = text.count('?')

    if n_q == 1:
        qi = text.index('?')
        suffix = text[qi + 1:].strip()
        text = text[:qi + 1].strip()

    elif n_q == 2:
        first_q = text.index('?')
        prefix = text[:first_q + 1].strip()

        text = text[first_q + 1:].strip()
        second_q = text.index('?')

        suffix = text[second_q + 1:].strip()
        text = text[:second_q + 1].strip()

    return (prefix or None), text, (suffix or None)


def clean_text(text, is_prompt=True):
    if(is_prompt):
        text = split_prompt(text)[1]
    text = text.lower()
    #text = re.sub(r'[^a-z0-9\s]', ' ', text) for pretrained
    #text = re.sub(r'\s+', ' ', text)
    return text.strip()


def tokenize(text, is_prompt=True):
    return clean_text(text, is_prompt).split()


def ap_at_3(actual, predicted):
    for i, p in enumerate(predicted[:3]):
        if p == actual:
            return 1.0 / (i + 1)
    return 0.0


def map_at_3(actuals, predictions):
    return float(np.mean([ap_at_3(a, p) for a, p in zip(actuals, predictions)]))


def make_submission(ids, predictions):
    return pd.DataFrame({
        'id': ids,
        'prediction': [' '.join(p[:3]) for p in predictions]
    })


def prepare_model_data(train_source, test_source):
    train_data = train_source.copy()
    test_data = test_source.copy()

    train_data["q_clean"] = train_data["prompt"].apply(clean_text)
    test_data["q_clean"] = test_data["prompt"].apply(clean_text)

    for option in OPTIONS:
        train_data[f"{option}_clean"] = train_data[option].apply(lambda text: clean_text(text, is_prompt=False))
        test_data[f"{option}_clean"] = test_data[option].apply(lambda text: clean_text(text, is_prompt=False))

    return train_data, test_data


def make_question_folds(clean_qs, folds=5):
    unique_qs = clean_qs.unique()
    shuffled_qs = unique_qs.copy()

    rng = np.random.default_rng(SEED)
    rng.shuffle(shuffled_qs)

    question_folds = np.array_split(shuffled_qs, folds)

    for fold, questions in enumerate(question_folds):
        print(f"Fold {fold}: {len(questions)} validation questions")

    return unique_qs, question_folds

# LOGISTIC REGRESSION

In [35]:
def make_lr_pairs(source_data, separator=" "):
    pair_rows = []

    for row in source_data.itertuples(index=False):
        for option in OPTIONS:
            pair_rows.append({
                    "id": row.id,
                    "option": option,
                    "combined": (row.q_clean + separator + getattr(row, f"{option}_clean")),
                    "label": (int(row.answer == option) if hasattr(row, "answer") else None)
                })

    return pd.DataFrame(pair_rows)


def rank_lr_predictions(pair_data, probabilities):
    ranked_data = pair_data.copy()
    ranked_data["probability"] = probabilities

    prediction_ids = []
    predictions = []

    for question_id, group in ranked_data.groupby("id", sort=False):
        ranked_options = group.sort_values("probability", ascending=False)["option"].tolist()

        prediction_ids.append(question_id)
        predictions.append(ranked_options[:3])

    return prediction_ids, predictions

## TFIDF + LOGISTIC REGRESSION

In [36]:
def run_tfidf_lr(train_source, test_source):
    train_data, test_data = prepare_model_data(train_source, test_source)

    train_pairs = make_lr_pairs(train_data)
    test_pairs = make_lr_pairs(test_data)

    print(train_pairs.head(5))
    print("Example pair:", train_pairs["combined"].iloc[0])

    vectorizer = TfidfVectorizer(max_features=20_000, ngram_range=(1, 2), sublinear_tf=True)

    train_features = vectorizer.fit_transform(train_pairs["combined"])
    train_labels = train_pairs["label"].values
    test_features = vectorizer.transform(test_pairs["combined"])

    groups = train_pairs["id"].values
    group_kfold = GroupKFold(n_splits=5)
    fold_scores = []

    model = LogisticRegression(max_iter=1_000, C=1.0, class_weight={0: 1, 1: 3})

    for fold, (train_indices, validation_indices) in enumerate(group_kfold.split(train_features, train_labels, groups)):
        model.fit(train_features[train_indices], train_labels[train_indices])

        validation_pairs = train_pairs.iloc[validation_indices].copy()
        validation_probabilities = model.predict_proba(train_features[validation_indices])[:, 1]

        _, validation_predictions = rank_lr_predictions(validation_pairs, validation_probabilities)

        validation_actuals = [train_data.loc[train_data["id"] == question_id, "answer"].iloc[0] for question_id in validation_pairs["id"].drop_duplicates()]

        score = map_at_3(validation_actuals, validation_predictions)
        fold_scores.append(score)
        print(f"Fold {fold} MAP@3: {score:.4f}")

    print(f"Mean MAP@3: {np.mean(fold_scores):.4f} +/- {np.std(fold_scores):.4f}")

    model.fit(train_features, train_labels)
    test_probabilities = model.predict_proba(test_features)[:, 1]

    return rank_lr_predictions(test_pairs, test_probabilities)

## TF-IDF + Logistic Regression LEARNINGS.MD

### Why did LR worked giving reasonably good score even if it doesn't have understanding?
- TF-IDF + LR has no concept of meaning.
- Rather it exploits the fact that the correct option tends to share more words/bigrams with the question than the wrong ones do.
- And it is proven because of roughly ~2x increase in the score from ~0.35 from random/base submission to ~0.75 with this.
- Helped clear the cutoff xD.

### Why did class weight worked inversely, so slightly?
- I overlooked a mistake and assigned class weights of {0:3, 1:1}, when infact label 0 makes 80% of the data, while label 1 makes 20%(Unfolded the options to individuals for Logistic regression to be trained upon).
- I inverted it but before I did so, I submitted it on kaggle and it raised a question.
- The faulty class weight gave a score of 0.7518 while the correct(better {0:1, 1:3}) gave slighlty less 0.7468, why?
- Turns out the ranking is mostly invariant to class weights and it is must certainly be noise.

## BERT CLS Pooling + Logistic Regression

In [37]:
def run_bert_lr(train_source, test_source):
    model_name = "bert-base-uncased"

    tokenizer = AutoTokenizer.from_pretrained(model_name)
    bert = AutoModel.from_pretrained(model_name).to(device)
    bert.eval()

    @torch.no_grad()
    def embed_texts(texts, batch_size=32):
        vectors = []

        for start in range(0, len(texts), batch_size):
            batch = texts[start:start + batch_size]
            encoded = tokenizer(
                batch,
                padding=True,
                truncation=True,
                max_length=512,
                return_tensors="pt"
            ).to(device)

            outputs = bert(**encoded)
            cls_vectors = outputs.last_hidden_state[:, 0]
            cls_vectors = functional.normalize(cls_vectors, dim=1)
            vectors.append(cls_vectors.cpu())

        return torch.cat(vectors).numpy()

    train_data, test_data = prepare_model_data(train_source, test_source)

    train_pairs = make_lr_pairs(train_data, separator=" [SEP] ")
    test_pairs = make_lr_pairs(test_data, separator=" [SEP] ")

    print(train_pairs.head(5))
    print("Example pair:", train_pairs["combined"].iloc[0])

    train_features = embed_texts(train_pairs["combined"].tolist())
    train_labels = train_pairs["label"].values

    groups = train_pairs["id"].values
    group_kfold = GroupKFold(n_splits=5)
    fold_scores = []

    model = LogisticRegression(max_iter=3_000, C=3, class_weight={0: 1, 1: 4})

    for fold, (train_indices, validation_indices) in enumerate(group_kfold.split(train_features, train_labels, groups)):
        model.fit(train_features[train_indices], train_labels[train_indices])

        validation_pairs = train_pairs.iloc[validation_indices].copy()
        validation_probabilities = model.predict_proba(train_features[validation_indices])[:, 1]

        _, validation_predictions = rank_lr_predictions(validation_pairs, validation_probabilities)

        validation_actuals = [train_data.loc[train_data["id"] == question_id, "answer"].iloc[0] for question_id in validation_pairs["id"].drop_duplicates()]

        score = map_at_3(validation_actuals, validation_predictions)
        fold_scores.append(score)
        print(f"Fold {fold} MAP@3: {score:.4f}")

    print(f"Mean MAP@3: {np.mean(fold_scores):.4f} +/- {np.std(fold_scores):.4f}")

    model.fit(train_features, train_labels)

    test_features = embed_texts(test_pairs["combined"].tolist())
    test_probabilities = model.predict_proba(test_features)[:, 1]

    return rank_lr_predictions(test_pairs, test_probabilities)

## BERT + LR LEARNINGS.MD

### Why did BERT+LR score less than TFIDF+LR?
- BERT embeddings have semantic meaning/understanding unlike tfidfs.
- It scored lower because it might have struggled as in the options words used are semantically similar.


# BiLSTM FROM SCRATCH
### with K-Folds ensembling

In [38]:
class BiLSTMVocabulary:
    def __init__(self):
        self.token_to_id = {
            "<PAD>": 0,
            "<UNK>": 1,
            "<SEP>": 2
        }
        self.id_to_token = {
            0: "<PAD>",
            1: "<UNK>",
            2: "<SEP>"
        }

    def build(self, token_lists):
        token_counts = Counter()

        for tokens in token_lists:
            token_counts.update(tokens)

        for token, _ in token_counts.most_common():
            if token in self.token_to_id:
                continue

            token_id = len(self.token_to_id)
            self.token_to_id[token] = token_id
            self.id_to_token[token_id] = token

    def encode(self, tokens, maximum_length):
        tokens = tokens[:maximum_length]
        token_ids = [self.token_to_id.get(token, self.token_to_id["<UNK>"]) for token in tokens]

        real_length = len(token_ids)
        token_ids += [self.token_to_id["<PAD>"]] * (maximum_length - real_length)
        return token_ids, real_length

    def __len__(self):
        return len(self.token_to_id)

In [39]:
def make_bilstm_pairs(source_data, has_labels=True):
    pair_rows = []

    for row in source_data.itertuples(index=False):
        question_tokens = row.q_clean.split()

        for option in OPTIONS:
            option_tokens = getattr(row, f"{option}_clean").split()
            tokens = question_tokens + ["<SEP>"] + option_tokens
            pair_row = {
                "id": int(row.id),
                "option": option,
                "tokens": tokens,
                "length": len(tokens),
            }
            if has_labels:
                pair_row["label"] = int(row.answer == option)

            pair_rows.append(pair_row)

    return pd.DataFrame(pair_rows)


def encode_bilstm_groups(pair_data,vocabulary, maximum_length, has_labels=True):
    samples = []

    for question_id, group in pair_data.groupby("id", sort=False):
        group = group.sort_values("option")
        input_ids, lengths = [], []

        for row in group.itertuples(index=False):
            row_ids, row_length = vocabulary.encode(row.tokens, maximum_length)
            input_ids.append(row_ids)
            lengths.append(row_length)

        sample = {
            "input_ids": torch.tensor(input_ids, dtype=torch.long),
            "length": torch.tensor(lengths, dtype=torch.long),
            "id": question_id,
        }

        if has_labels:
            label_index = group["label"].tolist().index(1)
            sample["label"] = torch.tensor(label_index, dtype=torch.long)

        samples.append(sample)

    return samples

In [40]:
def create_bilstm_model_class():
    import torch
    import torch.nn as nn

    class BiLSTMMCQ(nn.Module):
        def __init__(self, vocabulary_size, padding_index, embedding_dimension=128, hidden_dimension=128, number_of_layers=1, dropout=0.3):
            super().__init__()

            self.padding_index = padding_index
            self.embedding = nn.Embedding(vocabulary_size, embedding_dimension, padding_index)
            self.embedding_dropout = nn.Dropout(dropout)

            self.lstm = nn.LSTM(
                input_size=embedding_dimension,
                hidden_size=hidden_dimension,
                num_layers=number_of_layers,
                batch_first=True,
                bidirectional=True,
                dropout=dropout if number_of_layers > 1 else 0.0
            )

            self.classifier = nn.Sequential(
                nn.Linear(hidden_dimension * 2, hidden_dimension),
                nn.LayerNorm(hidden_dimension),
                nn.ReLU(),
                nn.Dropout(dropout),
                nn.Linear(hidden_dimension, 1)
            )

        def forward(self, input_ids, lengths):
            batch_size, number_of_options, sequence_length = input_ids.shape

            input_ids = input_ids.view(batch_size * number_of_options, sequence_length)
            lengths = lengths.view(batch_size * number_of_options)

            embedded = self.embedding(input_ids)
            embedded = self.embedding_dropout(embedded)

            packed = nn.utils.rnn.pack_padded_sequence(
                embedded,
                lengths.cpu(),
                batch_first=True,
                enforce_sorted=False
            )

            packed_output, _ = self.lstm(packed)

            output, _ = nn.utils.rnn.pad_packed_sequence(
                packed_output,
                batch_first=True,
                total_length=sequence_length
            )

            mask = ((input_ids != self.padding_index).unsqueeze(-1).to(output.dtype))
            pooled_output = ((output * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1.0))

            logits = self.classifier(pooled_output).squeeze(-1)
            return logits.view(batch_size, number_of_options)

    return BiLSTMMCQ

In [41]:
def run_bilstm(train_source, test_source):
    number_of_folds = 5
    maximum_length = 100
    batch_size = 64
    learning_rate = 1e-3
    epochs = 150
    patience = 30

    train_data, test_data = prepare_model_data(train_source, test_source)
    unique_questions, question_folds = make_question_folds(train_data["q_clean"], number_of_folds)

    print("Empty questions:", (train_data["q_clean"].str.len() == 0).sum())
    for option in OPTIONS:
        print(option, (train_data[f"{option}_clean"].str.len() == 0).sum())

    test_pairs = make_bilstm_pairs(test_data, has_labels=False)
    criterion = nn.CrossEntropyLoss()
    BiLSTMMCQ = create_bilstm_model_class()

    def train_epoch(model, loader, optimizer):
        model.train()
        total_loss, total_samples = 0.0, 0

        for batch in loader:
            input_ids = batch["input_ids"].to(device)
            lengths = batch["length"]
            labels = batch["label"].to(device)

            optimizer.zero_grad(set_to_none=True)
            logits = model(input_ids, lengths)
            loss = criterion(logits, labels)
            loss.backward()

            nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

            current_batch_size = input_ids.size(0)
            total_loss += loss.item() * current_batch_size
            total_samples += current_batch_size

        return total_loss / total_samples

    @torch.no_grad()
    def evaluate(model, loader):
        model.eval()
        total_loss, total_samples = 0.0, 0
        predictions, actual_answers = [], []

        for batch in loader:
            input_ids = batch["input_ids"].to(device)
            lengths = batch["length"]
            labels = batch["label"].to(device)

            logits = model(input_ids, lengths)
            loss = criterion(logits, labels)
            probabilities = torch.softmax(logits, dim=1).cpu().numpy()

            current_batch_size = input_ids.size(0)
            total_loss += loss.item() * current_batch_size
            total_samples += current_batch_size

            for probability_row, label in zip(probabilities, labels.cpu().numpy()):
                ranked_options = [OPTIONS[index] for index in np.argsort(-probability_row)]
                predictions.append(ranked_options[:3])
                actual_answers.append(OPTIONS[label])

        return {
            "loss": total_loss / total_samples,
            "map3": map_at_3(actual_answers, predictions),
            "predictions": predictions,
            "actual_answers": actual_answers
        }

    @torch.no_grad()
    def predict_test(model, loader):
        model.eval()
        ids, probabilities = [], []

        for batch in loader:
            input_ids = batch["input_ids"].to(device)
            lengths = batch["length"]

            logits = model(input_ids, lengths)
            batch_probabilities = torch.softmax(logits, dim=1).cpu().numpy()

            ids.extend(batch["id"].cpu().numpy().tolist())
            probabilities.extend(batch_probabilities)

        return ids, probabilities

    fold_results = []

    for fold in range(number_of_folds):
        validation_questions = set(question_folds[fold])
        training_questions = set(unique_questions) - validation_questions

        fold_train = train_data[train_data["q_clean"].isin(training_questions)].reset_index(drop=True)
        fold_validation = train_data[train_data["q_clean"].isin(validation_questions)].reset_index(drop=True)

        train_pairs = make_bilstm_pairs(fold_train, has_labels=True)
        validation_pairs = make_bilstm_pairs(fold_validation, has_labels=True)

        vocabulary = BiLSTMVocabulary()
        vocabulary.build(train_pairs["tokens"])

        train_samples = encode_bilstm_groups(train_pairs, vocabulary, maximum_length, has_labels=True)
        validation_samples = encode_bilstm_groups(validation_pairs, vocabulary, maximum_length, has_labels=True)

        train_loader = DataLoader(train_samples, batch_size=batch_size, shuffle=True)
        validation_loader = DataLoader(validation_samples, batch_size=batch_size * 2, shuffle=False)

        model = BiLSTMMCQ(
            vocabulary_size=len(vocabulary),
            padding_index=0,
            embedding_dimension=128,
            hidden_dimension=128,
            number_of_layers=1,
            dropout=0.3
        ).to(device)

        optimizer = Adam(model.parameters(), lr=learning_rate)

        best_map3 = -1.0
        best_epoch = 0
        epochs_without_improvement = 0
        history = []
        checkpoint_path = f"best_bilstm_fold{fold}.pt"

        for epoch in range(1, epochs + 1):
            training_loss = train_epoch(model, train_loader, optimizer)
            validation_result = evaluate(model, validation_loader)

            validation_loss = validation_result["loss"]
            validation_map3 = validation_result["map3"]

            history.append({
                "epoch": epoch,
                "train_loss": training_loss,
                "val_loss": validation_loss,
                "val_map3": validation_map3
                })

            print(f"Fold {fold} | Epoch {epoch} | Train loss: {training_loss:.4f} | Val loss: {validation_loss:.4f} | Val MAP@3: {validation_map3:.4f}")

            if validation_map3 > best_map3:
                best_map3 = validation_map3
                best_epoch = epoch
                epochs_without_improvement = 0
                torch.save(model.state_dict(), checkpoint_path)
                print(f"Fold {fold}: Best model saved: MAP@3={best_map3:.4f}")
            else:
                epochs_without_improvement += 1
                if epochs_without_improvement >= patience:
                    print(f"Early stopping after epoch {epoch}.")
                    break

        print(f"\nFold {fold} training complete")
        print(f"Fold {fold}: Best epoch: {best_epoch}")
        print(f"Fold {fold}: Best validation MAP@3: {best_map3}\n")

        model.load_state_dict(torch.load(checkpoint_path, map_location=device, weights_only=True))
        model.eval()

        best_validation_result = evaluate(model, validation_loader)

        test_samples = encode_bilstm_groups(test_pairs, vocabulary, maximum_length, has_labels=False)
        test_loader = DataLoader(test_samples, batch_size=batch_size * 2, shuffle=False)
        test_ids, test_probabilities = predict_test(model, test_loader)

        fold_results.append(
            {
                "fold": fold,
                "history": pd.DataFrame(history),
                "best_map3": best_map3,
                "oof_predictions": best_validation_result["predictions"],
                "oof_actuals": best_validation_result["actual_answers"],
                "test_ids": test_ids,
                "test_probabilities": np.array(test_probabilities)
            }
        )

        del model
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    oof_predictions, oof_actuals = [], []
    for result in fold_results:
        oof_predictions.extend(result["oof_predictions"])
        oof_actuals.extend(result["oof_actuals"])
        print(f"Fold {result['fold']} best validation MAP@3: {result['best_map3']:.4f}")

    overall_map3 = map_at_3(oof_actuals, oof_predictions)
    print("Overall OOF MAP@3:", overall_map3)

    history_data = pd.concat([result["history"].assign(fold=result["fold"]) for result in fold_results], ignore_index=True)
    print(history_data.tail())

    ensemble_probabilities = np.mean([result["test_probabilities"] for result in fold_results], axis=0)
    test_predictions = [[OPTIONS[index] for index in np.argsort(-probability_row)[:3]] for probability_row in ensemble_probabilities]

    return fold_results[0]["test_ids"], test_predictions

## BiLSTM from scratch LEARNINGS.MD
- built entire BiLSTM from scratch with bit of help
- the kaggle score vastly outperformed val score, meaning our val score is unreliable
- fix will be to implement k-folds cross validation
- current score disparity MAP@3 is `Val: 0.59` and `Kaggle: 0.71`

## BiLSTM with KfoldsEnsemble and Softmax
- Switched from BCE to softmax with Cross entropy loss.
- Before train test split was 80/20 only, now we made 5 folds using np.random and then we train them, and later use ensemble technique on it, to prevent a single improper split.
- Score on Kaggle is `0.75`

# DeBERTa & MiniLM

In [42]:
TRANSFORMER_CONFIGS = {
    "deberta": {
        "model_name": "microsoft/deberta-v3-small",
        "batch_size": 8,
        "learning_rate": 2e-5,
        "checkpoint_prefix": "best_deberta"
    },
    "minilm": {
        "model_name": "sentence-transformers/all-MiniLM-L6-v2",
        "batch_size": 32,
        "learning_rate": 5e-5,
        "checkpoint_prefix": "best_minilm"
    }
}

TRANSFORMER_MAX_LENGTH = 100
TRANSFORMER_FOLDS = 5
TRANSFORMER_EPOCHS = 15
TRANSFORMER_PATIENCE = 5

In [43]:
def make_transformer_pairs(source_data, has_labels=True):
    pair_rows = []

    for row in source_data.itertuples(index=False):
        for option in OPTIONS:
            pair_row = {
                "id": int(row.id),
                "option": option,
                "question": row.q_clean,
                "option_text": getattr(row, f"{option}_clean")
            }
            if has_labels:
                pair_row["label"] = int(row.answer == option)
            pair_rows.append(pair_row)

    return pd.DataFrame(pair_rows)


def encode_transformer_groups(pair_data, tokenizer, maximum_length, has_labels=True):
    samples = []

    for question_id, group in pair_data.groupby("id", sort=False):
        group = group.sort_values("option")

        encoded = tokenizer(
            group["question"].tolist(),
            group["option_text"].tolist(),
            truncation=True,
            max_length=maximum_length,
            padding="max_length",
            return_tensors="pt"
        )

        sample = {
            "input_ids": encoded["input_ids"],
            "attention_mask": encoded["attention_mask"],
            "id": question_id
        }

        if has_labels:
            label_index = group["label"].tolist().index(1)
            sample["label"] = torch.tensor(label_index, dtype=torch.long)

        samples.append(sample)

    return samples

In [44]:
def run_transformer_mcq(train_source, test_source, transformer_choice):
    if transformer_choice not in TRANSFORMER_CONFIGS:
        raise ValueError("transformer_choice must be either 'deberta' or 'minilm'.")

    config = TRANSFORMER_CONFIGS[transformer_choice]
    model_name = config["model_name"]
    batch_size = config["batch_size"]
    learning_rate = config["learning_rate"]

    print("Transformer selection:", transformer_choice)
    print("Model checkpoint:", model_name)
    print("Device:", device)
    print("Batch size:", batch_size)
    print("Learning rate:", learning_rate)

    train_data, test_data = prepare_model_data(train_source, test_source)
    unique_questions, question_folds = make_question_folds(train_data["q_clean"], TRANSFORMER_FOLDS)

    print("Empty questions:", (train_data["q_clean"].str.len() == 0).sum())
    for option in OPTIONS:
        print(option, (train_data[f"{option}_clean"].str.len() == 0).sum())

    tokenizer = AutoTokenizer.from_pretrained(model_name)
    test_pairs = make_transformer_pairs(test_data, has_labels=False)
    test_samples = encode_transformer_groups(test_pairs, tokenizer, TRANSFORMER_MAX_LENGTH, has_labels=False)

    def create_model():
        return AutoModelForMultipleChoice.from_pretrained(model_name, torch_dtype=torch.float32)

    def train_epoch(model, loader, optimizer, scheduler):
        model.train()
        total_loss, total_samples = 0.0, 0

        for batch in loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["label"].to(device)

            optimizer.zero_grad(set_to_none=True)
            outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            loss = outputs.loss
            loss.backward()

            nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            scheduler.step()

            current_batch_size = input_ids.size(0)
            total_loss += loss.item() * current_batch_size
            total_samples += current_batch_size

        return total_loss / total_samples

    @torch.no_grad()
    def evaluate(model, loader):
        model.eval()
        total_loss, total_samples = 0.0, 0
        predictions, actual_answers = [], []

        for batch in loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["label"].to(device)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            loss = outputs.loss
            probabilities = torch.softmax(outputs.logits, dim=1).cpu().numpy()

            current_batch_size = input_ids.size(0)
            total_loss += loss.item() * current_batch_size
            total_samples += current_batch_size

            for probability_row, label in zip(probabilities, labels.cpu().numpy()):
                ranked_options = [OPTIONS[index] for index in np.argsort(-probability_row)]
                predictions.append(ranked_options[:3])
                actual_answers.append(OPTIONS[label])

        return {
            "loss": total_loss / total_samples,
            "map3": map_at_3(actual_answers, predictions),
            "predictions": predictions,
            "actual_answers": actual_answers
        }

    @torch.no_grad()
    def predict_test(model, loader):
        model.eval()
        ids, probabilities = [], []

        for batch in loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)

            logits = model(input_ids=input_ids, attention_mask=attention_mask).logits
            batch_probabilities = torch.softmax(logits, dim=1).cpu().numpy()

            ids.extend(batch["id"].cpu().numpy().tolist())
            probabilities.extend(batch_probabilities)

        return ids, probabilities

    fold_results = []

    for fold in range(TRANSFORMER_FOLDS):
        validation_questions = set(question_folds[fold])
        training_questions = set(unique_questions) - validation_questions

        fold_train = train_data[train_data["q_clean"].isin(training_questions)].reset_index(drop=True)
        fold_validation = train_data[train_data["q_clean"].isin(validation_questions)].reset_index(drop=True)

        train_pairs = make_transformer_pairs(fold_train, has_labels=True)
        validation_pairs = make_transformer_pairs(fold_validation, has_labels=True)

        train_samples = encode_transformer_groups(train_pairs, tokenizer, TRANSFORMER_MAX_LENGTH, has_labels=True)
        validation_samples = encode_transformer_groups(validation_pairs, tokenizer, TRANSFORMER_MAX_LENGTH, has_labels=True)

        train_loader = DataLoader(train_samples, batch_size=batch_size, shuffle=True)
        validation_loader = DataLoader(validation_samples, batch_size=batch_size * 2, shuffle=False)

        model = create_model().to(device)
        optimizer = AdamW(model.parameters(), lr=learning_rate, weight_decay=0.01)

        total_steps = len(train_loader) * TRANSFORMER_EPOCHS
        scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=int(0.1 * total_steps), num_training_steps=total_steps)

        best_map3 = -1.0
        best_epoch = 0
        epochs_without_improvement = 0
        history = []
        checkpoint_path = (f"{config['checkpoint_prefix']}_fold{fold}.pt")

        for epoch in range(1, TRANSFORMER_EPOCHS + 1):
            training_loss = train_epoch(model, train_loader, optimizer, scheduler)
            validation_result = evaluate(model, validation_loader)

            validation_loss = validation_result["loss"]
            validation_map3 = validation_result["map3"]

            history.append({
                "epoch": epoch,
                "train_loss": training_loss,
                "val_loss": validation_loss,
                "val_map3": validation_map3,
            })

            print(f"Fold {fold}: Epoch {epoch} | Train loss: {training_loss:.4f} | Val loss: {validation_loss:.4f} | Val MAP@3: {validation_map3:.4f}")

            if validation_map3 > best_map3:
                best_map3 = validation_map3
                best_epoch = epoch
                epochs_without_improvement = 0
                torch.save(model.state_dict(), checkpoint_path)
                print(f"Fold {fold}: Best model saved: MAP@3={best_map3:.4f}")
            else:
                epochs_without_improvement += 1
                if (epochs_without_improvement >= TRANSFORMER_PATIENCE):
                    print(f"Early stopping after epoch {epoch}.")
                    break

        print(f"\nFold {fold} training complete")
        print(f"Fold {fold}: Best epoch: {best_epoch}")
        print(f"Fold {fold}: Best validation MAP@3: {best_map3}\n")

        model.load_state_dict(torch.load(checkpoint_path, map_location=device, weights_only=True))
        model.eval()

        best_validation_result = evaluate(model, validation_loader)

        test_loader = DataLoader(test_samples, batch_size=batch_size * 2, shuffle=False)
        test_ids, test_probabilities = predict_test(model, test_loader)

        fold_results.append({
            "fold": fold,
            "history": pd.DataFrame(history),
            "best_map3": best_map3,
            "oof_predictions": best_validation_result["predictions"],
            "oof_actuals": best_validation_result["actual_answers"],
            "test_ids": test_ids,
            "test_probabilities": np.array(test_probabilities),
        })

        del model
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    oof_predictions, oof_actuals = [], []
    for result in fold_results:
        oof_predictions.extend(result["oof_predictions"])
        oof_actuals.extend(result["oof_actuals"])
        print(f"Fold {result['fold']} best validation MAP@3: {result['best_map3']:.4f}")

    overall_map3 = map_at_3(oof_actuals, oof_predictions)
    print("Overall OOF MAP@3:", overall_map3)

    history_data = pd.concat([result["history"].assign(fold=result["fold"]) for result in fold_results], ignore_index=True)
    print(history_data.tail())

    ensemble_probabilities = np.mean([result["test_probabilities"] for result in fold_results], axis=0)
    test_predictions = [[OPTIONS[index] for index in np.argsort(-probability_row)[:3]]  for probability_row in ensemble_probabilities]

    return fold_results[0]["test_ids"], test_predictions

## DeBERTa LEARNINGS.MD

### with K-Fold OOF
- Used pretrained `microsoft/deberta-v3-small` and finetuned it on question option pairs.
- Used CrossEntropyLoss over the 5 logits rather than binary classification.
- The 5 fold models are ensembled for test predictions.
- MAP@3: `Val: 0.647` and `Kaggle: 0.741`.

### with AutoModelForMultipleChoice
- Replaced the manually created AutoModel + CLS pooling + Linear architecture with Hugging Face's AutoModelForMultipleChoice.
- no significant increase in score, which suggests that the classification head was not a major bottleneck.
- MAP@3: `Val: 0.648` and `Kaggle: 0.743`.

### DeBERTa-base
- Takes quite bit of time and scores less than cutoof.
- Though interesting take of logits vs probabilities averaging.
- Kaggle MAP@3 for `Logits: 0.717` vs `Probabilities: 0.722`

## MiniLM LEARNINGS.MD
- Just replaced deberta small with minilm, got `0.745 MAP@3`

---
# SUBMISSION

In [ ]:
model_train = pd.read_csv(TRAIN_PATH)
model_test = pd.read_csv(TEST_PATH)

if FINAL_MODEL == "tfidf_lr":
    test_ids, test_predictions = run_tfidf_lr(model_train, model_test)
elif FINAL_MODEL == "bert_lr":
    test_ids, test_predictions = run_bert_lr(model_train, model_test)
elif FINAL_MODEL == "bilstm":
    test_ids, test_predictions = run_bilstm(model_train, model_test)
elif FINAL_MODEL in {"deberta", "minilm"}:
    test_ids, test_predictions = run_transformer_mcq(model_train, model_test, transformer_choice=FINAL_MODEL)

print("Number of test IDs:", len(test_ids))
print("Number of predictions:", len(test_predictions))
print("First prediction:", test_predictions[0])

Transformer selection: deberta
Model checkpoint: microsoft/deberta-v3-small
Device: cuda
Batch size: 8
Learning rate: 2e-05
Fold 0: 51 validation questions  
Fold 1: 51 validation questions  
Fold 2: 50 validation questions  
Fold 3: 50 validation questions  
Fold 4: 50 validation questions  
Empty questions: 0
A 0
B 0
C 0
D 0
E 0


Fold 0: Epoch 1 | Train loss: 1.5656 | Val loss: 1.5353 | Val MAP@3: 0.4416  
Fold 0: Best model saved: MAP@3=0.4416  
Fold 0: Epoch 2 | Train loss: 0.9344 | Val loss: 1.2251 | Val MAP@3: 0.6289  
Fold 0: Best model saved: MAP@3=0.6289  
Fold 0: Epoch 3 | Train loss: 0.3491 | Val loss: 1.3914 | Val MAP@3: 0.5863  
Fold 0: Epoch 4 | Train loss: 0.1097 | Val loss: 1.5807 | Val MAP@3: 0.6307  
Fold 0: Best model saved: MAP@3=0.6307  
Fold 0: Epoch 5 | Train loss: 0.0437 | Val loss: 1.5453 | Val MAP@3: 0.5988  
Fold 0: Epoch 6 | Train loss: 0.0160 | Val loss: 1.8663 | Val MAP@3: 0.5800  
Fold 0: Epoch 7 | Train loss: 0.0080 | Val loss: 1.8590 | Val MAP@3: 0.6182  
Fold 0: Epoch 8 | Train loss: 0.0058 | Val loss: 1.6337 | Val MAP@3: 0.6043  
Fold 0: Epoch 9 | Train loss: 0.0026 | Val loss: 1.9614 | Val MAP@3: 0.6645  
Fold 0: Best model saved: MAP@3=0.6645  
Fold 0: Epoch 10 | Train loss: 0.0005 | Val loss: 1.9228 | Val MAP@3: 0.5951  
Fold 0: Epoch 11 | Train loss: 0.0039 | Val loss: 2.0835 | Val MAP@3: 0.5954  
Fold 0: Epoch 12 | Train loss: 0.0009 | Val loss: 2.1627 | Val MAP@3: 0.6289  
Fold 0: Epoch 13 | Train loss: 0.0002 | Val loss: 2.0499 | Val MAP@3: 0.6226  
Fold 0: Epoch 14 | Train loss: 0.0001 | Val loss: 2.0166 | Val MAP@3: 0.6252  
Early stopping after epoch 14.

Fold 0 training complete  
Fold 0: Best epoch: 9  
Fold 0: Best validation MAP@3: 0.6644640234948604  


Fold 1: Epoch 1 | Train loss: 1.5587 | Val loss: 1.4407 | Val MAP@3: 0.5302  
Fold 1: Best model saved: MAP@3=0.5302  
Fold 1: Epoch 2 | Train loss: 0.9710 | Val loss: 1.3820 | Val MAP@3: 0.6172  
Fold 1: Best model saved: MAP@3=0.6172  
Fold 1: Epoch 3 | Train loss: 0.4178 | Val loss: 1.5464 | Val MAP@3: 0.6471  
Fold 1: Best model saved: MAP@3=0.6471  
Fold 1: Epoch 4 | Train loss: 0.1555 | Val loss: 1.8179 | Val MAP@3: 0.6908  
Fold 1: Best model saved: MAP@3=0.6908  
Fold 1: Epoch 5 | Train loss: 0.0698 | Val loss: 1.8827 | Val MAP@3: 0.6413  
Fold 1: Epoch 6 | Train loss: 0.0230 | Val loss: 1.9500 | Val MAP@3: 0.6916  
Fold 1: Best model saved: MAP@3=0.6916  
Fold 1: Epoch 7 | Train loss: 0.0127 | Val loss: 1.9495 | Val MAP@3: 0.6842  
Fold 1: Epoch 8 | Train loss: 0.0037 | Val loss: 2.0936 | Val MAP@3: 0.6270  
Fold 1: Epoch 9 | Train loss: 0.0033 | Val loss: 2.1171 | Val MAP@3: 0.6634  
Fold 1: Epoch 10 | Train loss: 0.0028 | Val loss: 2.1046 | Val MAP@3: 0.6875  
Fold 1: Epoch 11 | Train loss: 0.0003 | Val loss: 2.0800 | Val MAP@3: 0.6446  
Early stopping after epoch 11.

Fold 1 training complete  
Fold 1: Best epoch: 6  
Fold 1: Best validation MAP@3: 0.6915849673202613  


Fold 2: Epoch 1 | Train loss: 1.5190 | Val loss: 1.4844 | Val MAP@3: 0.4831  
Fold 2: Best model saved: MAP@3=0.4831  
Fold 2: Epoch 2 | Train loss: 0.8667 | Val loss: 1.6282 | Val MAP@3: 0.6027  
Fold 2: Best model saved: MAP@3=0.6027  
Fold 2: Epoch 3 | Train loss: 0.3752 | Val loss: 2.3410 | Val MAP@3: 0.5525  
Fold 2: Epoch 4 | Train loss: 0.1669 | Val loss: 3.0000 | Val MAP@3: 0.5467  
Fold 2: Epoch 5 | Train loss: 0.0679 | Val loss: 2.6729 | Val MAP@3: 0.5184  
Fold 2: Epoch 6 | Train loss: 0.0291 | Val loss: 3.8286 | Val MAP@3: 0.5216  
Fold 2: Epoch 7 | Train loss: 0.0095 | Val loss: 2.5156 | Val MAP@3: 0.5024  
Early stopping after epoch 7.

Fold 2 training complete  
Fold 2: Best epoch: 2  
Fold 2: Best validation MAP@3: 0.6027450980392157  


Fold 3: Epoch 1 | Train loss: 1.5345 | Val loss: 1.4025 | Val MAP@3: 0.5037  
Fold 3: Best model saved: MAP@3=0.5037  
Fold 3: Epoch 2 | Train loss: 0.8139 | Val loss: 1.7415 | Val MAP@3: 0.5292  
Fold 3: Best model saved: MAP@3=0.5292  
Fold 3: Epoch 3 | Train loss: 0.3189 | Val loss: 2.0835 | Val MAP@3: 0.5260  
Fold 3: Epoch 4 | Train loss: 0.1238 | Val loss: 2.1095 | Val MAP@3: 0.5487  
Fold 3: Best model saved: MAP@3=0.5487  
Fold 3: Epoch 5 | Train loss: 0.0362 | Val loss: 2.9513 | Val MAP@3: 0.5065  
Fold 3: Epoch 6 | Train loss: 0.0177 | Val loss: 1.9689 | Val MAP@3: 0.4991  
Fold 3: Epoch 7 | Train loss: 0.0100 | Val loss: 2.3036 | Val MAP@3: 0.5599  
Fold 3: Best model saved: MAP@3=0.5599  
Fold 3: Epoch 8 | Train loss: 0.0016 | Val loss: 2.8704 | Val MAP@3: 0.5580  
Fold 3: Epoch 9 | Train loss: 0.0033 | Val loss: 2.9845 | Val MAP@3: 0.5501  
Fold 3: Epoch 10 | Train loss: 0.0011 | Val loss: 3.1055 | Val MAP@3: 0.5395  
Fold 3: Epoch 11 | Train loss: 0.0006 | Val loss: 2.9784 | Val MAP@3: 0.5260  
Fold 3: Epoch 12 | Train loss: 0.0007 | Val loss: 3.2043 | Val MAP@3: 0.5357  
Early stopping after epoch 12.

Fold 3 training complete  
Fold 3: Best epoch: 7  
Fold 3: Best validation MAP@3: 0.5598885793871866  


Fold 4: Epoch 1 | Train loss: 1.5358 | Val loss: 1.3259 | Val MAP@3: 0.7053  
Fold 4: Best model saved: MAP@3=0.7053  
Fold 4: Epoch 2 | Train loss: 0.8514 | Val loss: 1.2405 | Val MAP@3: 0.6667  
Fold 4: Epoch 3 | Train loss: 0.3306 | Val loss: 1.1276 | Val MAP@3: 0.7222  
Fold 4: Best model saved: MAP@3=0.7222  
Fold 4: Epoch 4 | Train loss: 0.1449 | Val loss: 1.3135 | Val MAP@3: 0.6620  
Fold 4: Epoch 5 | Train loss: 0.0656 | Val loss: 1.3598 | Val MAP@3: 0.6648  
Fold 4: Epoch 6 | Train loss: 0.0339 | Val loss: 1.9428 | Val MAP@3: 0.6229  
Fold 4: Epoch 7 | Train loss: 0.0114 | Val loss: 1.7768 | Val MAP@3: 0.6398  
Fold 4: Epoch 8 | Train loss: 0.0082 | Val loss: 1.8650 | Val MAP@3: 0.6568  
Early stopping after epoch 8.

Fold 4 training complete  
Fold 4: Best epoch: 3  
Fold 4: Best validation MAP@3: 0.7222222222222222  

Fold 0 best validation MAP@3: 0.6645  
Fold 1 best validation MAP@3: 0.6916  
Fold 2 best validation MAP@3: 0.6027  
Fold 3 best validation MAP@3: 0.5599  
Fold 4 best validation MAP@3: 0.7222  
Overall OOF MAP@3: 0.6483333333333332
    epoch  train_loss  val_loss  val_map3  fold  
47      4    0.144889  1.313464  0.661959     4  
48      5    0.065624  1.359790  0.664783     4 
...  
51      8    0.008216  1.864951  0.656780     4  
Number of test IDs: 500  
Number of predictions: 500  
First prediction: ['A', 'C', 'B']  

In [46]:
submission = make_submission(test_ids, test_predictions)
submission.to_csv('submission.csv', index=False)

submission.head(5)

,id,prediction
0,1,A C B
1,2,B A E
2,3,B E D
3,4,E C D
4,5,C D A
